# Tracking Pipeline Notebook

This notebook demonstrates the cell tracking pipeline:
- Segmenting cells in consecutive frames
- Linking cells across frames using Hungarian algorithm
- Building lineage/trajectory information
- Detecting cell divisions

In [2]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging
from collections import defaultdict

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

plt.rcParams['figure.figsize'] = (14, 8)
np.set_printoptions(precision=3, suppress=True)

VOXEL_SIZE_UM = (1.625, 0.40625, 0.40625)

print("=" * 60)
print("SETUP INSTRUCTIONS")
print("=" * 60)
print("\nTo run this notebook, execute from project root:")
print("\n  pip install -e .")
print("  pip install zarr")
print("\nThen restart the Jupyter kernel.")
print("=" * 60)

# Try importing modules
try:
    from biohub_tracking.data.zarr_loader import iter_frames
    from biohub_tracking.segmentation.segmenter import CellSegmenter
    from biohub_tracking.tracking.linker import HungarianLinker
    from biohub_tracking.tracking.division_detector import DivisionDetector
    from biohub_tracking.tracking.lineage_builder import LineageBuilder
    MODULES_AVAILABLE = True
    print("✓ biohub_tracking modules available")
except ImportError as e:
    MODULES_AVAILABLE = False
    print(f"✗ biohub_tracking not available - {e}")

SETUP INSTRUCTIONS

To run this notebook, execute from project root:

  pip install -e .
  pip install zarr

Then restart the Jupyter kernel.
✗ biohub_tracking not available - No module named 'biohub_tracking'


## 1. Load and Segment Multiple Frames

In [ ]:
# Find sample data
data_dir = Path("../data")
sample_path = None

if (data_dir / "train").exists():
    samples = sorted((data_dir / "train").glob("*.zarr"))
    if samples:
        sample_path = samples[0]
elif (data_dir / "test").exists():
    samples = sorted((data_dir / "test").glob("*.zarr"))
    if samples:
        sample_path = samples[0]

if sample_path:
    print(f"Processing sample: {sample_path.name}")
    
    # Initialize segmenter
    segmenter = CellSegmenter(
        method="blob",
        min_size=30,
        anisotropy=VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1],
        voxel_size_um=VOXEL_SIZE_UM
    )
    
    # Segment first 5 frames
    all_cells = {}
    frame_count = 0
    
    for frame_index, image in iter_frames(sample_path):
        if frame_count >= 5:
            break
            
        labels, cells = segmenter.segment_frame(image, frame_index=frame_index)
        all_cells[frame_index] = cells
        print(f"Frame {frame_index}: {len(cells)} cells")
        frame_count += 1
    
    print(f"\nSegmented {frame_count} frames")
else:
    print("No sample data found")

## 2. Link Cells Across Frames

In [ ]:
if all_cells:
    # Initialize linker
    linker = HungarianLinker(
        max_distance=7.0,
        use_volume_cost=False
    )
    
    # Link consecutive frames
    links = []
    linked_ids = {}  # Frame -> [(source_id, target_id)]
    
    sorted_frames = sorted(all_cells.keys())
    print(f"Linking {len(sorted_frames)-1} consecutive frame pairs...\n")
    
    for frame_idx in range(len(sorted_frames)-1):
        frame1 = sorted_frames[frame_idx]
        frame2 = sorted_frames[frame_idx + 1]
        
        cells1 = all_cells[frame1]
        cells2 = all_cells[frame2]
        
        frame_links = linker.link(cells1, cells2)
        
        # Store links
        links.extend([
            (frame1, source, target, confidence)
            for source, target, confidence in frame_links
        ])
        linked_ids[frame1] = [(source, target) for source, target, _ in frame_links]
        
        print(f"Frames {frame1}->{frame2}: {len(frame_links)} links")
    
    print(f"\nTotal links: {len(links)}")
    
    # Print sample links
    if links:
        print("\nSample links (frame, source_id, target_id, confidence):")
        for link in links[:5]:
            print(f"  {link}")

## 3. Detect Cell Divisions

In [ ]:
if all_cells and linked_ids:
    # Initialize division classifier
    division_detector = DivisionDetector(
        max_distance_um=10.0
    )
    
    # Detect divisions
    divisions = division_detector.detect(all_cells, links)
    
    print(f"Division Detection Results:")
    print(f"Total potential divisions: {len(divisions)}")
    
    if divisions:
        print("\nDetected divisions:")
        for division in divisions[:10]:  # Show first 10
            print(f"  {division}")
    else:
        print("\nNo divisions detected in this sequence")

## 4. Build Lineage Graph

In [ ]:
if links:
    # Build lineage structure
    lineage_graph = defaultdict(list)
    reverse_links = defaultdict(list)  # target_id -> [source_ids]
    
    for frame, source_id, target_id, confidence in links:
        lineage_graph[source_id].append(target_id)
        reverse_links[target_id].append(source_id)
    
    # Analyze lineage structure
    print("Lineage Graph Statistics:")
    print(f"  Total cells in graph: {len(lineage_graph)}")
    
    # Find root cells (cells with no parents)
    all_cells_set = set(lineage_graph.keys())
    all_targets = set()
    for targets in lineage_graph.values():
        all_targets.update(targets)
    
    root_cells = all_cells_set - all_targets
    print(f"  Root cells (no parents): {len(root_cells)}")
    
    # Find leaf cells (cells with no children)
    leaf_cells = all_targets - all_cells_set
    print(f"  Leaf cells (no children): {len(leaf_cells)}")
    
    # Analyze lineage lengths
    trajectory_lengths = []
    visited = set()
    
    def get_lineage_length(cell_id):
        length = 1
        if cell_id in lineage_graph:
            for child in lineage_graph[cell_id]:
                length = max(length, 1 + get_lineage_length(child))
        return length
    
    for root_cell in root_cells:
        trajectory_lengths.append(get_lineage_length(root_cell))
    
    print(f"  Trajectory lengths: min={np.min(trajectory_lengths)}, "
          f"max={np.max(trajectory_lengths)}, mean={np.mean(trajectory_lengths):.1f}")
    
    # Plot trajectory length distribution
    if trajectory_lengths:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(trajectory_lengths, bins=20, edgecolor='black', alpha=0.7)
        ax.set_xlabel('Trajectory Length (frames)')
        ax.set_ylabel('Frequency')
        ax.set_title('Distribution of Cell Trajectory Lengths')
        ax.axvline(np.mean(trajectory_lengths), color='r', linestyle='--', 
                    label=f'Mean: {np.mean(trajectory_lengths):.1f}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()

## 5. Visualize Tracking Results

In [ ]:
# Summary statistics
print("Tracking Pipeline Summary:")
print("="*50)
print(f"Frames processed: {len(all_cells)}")
print(f"Total cells detected: {sum(len(cells) for cells in all_cells.values())}")
print(f"Total links created: {len(links)}")
if divisions:
    print(f"Divisions detected: {len(divisions)}")
else:
    print(f"Divisions detected: 0")
print(f"\nLinking algorithm: Hungarian")
print(f"Max linking distance: 7.0 µm")
print(f"Voxel size: {VOXEL_SIZE_UM} µm")
print(f"\nNext step: Evaluate tracking performance")